# Restormer NPU v1 Audit Notebook

This notebook is a self-contained audit/export notebook for the baseline Restormer snapshot under `Restormer NPU v1/`.

- It is not the canonical teacher training path.
- It is not the Lab 3 submission notebook.
- It preserves the required `256x256x3` input/output contract and writes ONNX, calibration, and MXQ handoff artifacts.


## Setup

This notebook keeps runtime imports local to `Restormer NPU v1/` and uses the same repo-level conventions for `lab3-data`, `lab3-runs`, and `runs/<started_day>/<run_name>/...`.


In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import torch
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / 'Data').is_dir() and (candidate / 'lab3_step2_onnx_to_mxq.py').exists():
        PROJECT_ROOT = candidate
        break
if not (PROJECT_ROOT / 'Data').is_dir():
    raise FileNotFoundError(f'Could not locate Lab3 project root from {NOTEBOOK_DIR}')
PACKAGE_ROOT = PROJECT_ROOT / 'Restormer NPU v1'
if not PACKAGE_ROOT.is_dir():
    raise FileNotFoundError(f'Expected package root at {PACKAGE_ROOT}')

TOOLS_ROOT = PACKAGE_ROOT / 'tools'
sys.path.insert(0, str(PACKAGE_ROOT))
sys.path.insert(0, str(TOOLS_ROOT))

from audit_support import (
    audit_onnx_graph,
    build_model,
    build_run_layout,
    export_calibration_dataset,
    export_to_onnx,
    load_config,
    operator_risk_markdown,
    resolve_profile,
    run_mxq_handoff,
    set_seed,
    validate_data_layout,
    verify_model_contract,
    write_json,
)

CONFIG_PATH = PACKAGE_ROOT / 'configs' / 'restormer_npu_v1.yaml'
DATA_ROOT = PROJECT_ROOT / 'Data'
MODAL_DATA_VOLUME = os.environ.get('LAB3_NOTEBOOK_MODAL_DATA_VOLUME', 'lab3-data')
MODAL_RUNS_VOLUME = os.environ.get('LAB3_NOTEBOOK_MODAL_RUNS_VOLUME', 'lab3-runs')
PROFILE = os.environ.get('RESTORMER_NPU_V1_PROFILE', '') or None
RUN_NAME = os.environ.get('RESTORMER_NPU_V1_RUN_NAME', 'restormer_npu_v1_smoke')
STARTED_DAY = os.environ.get('RESTORMER_NPU_V1_STARTED_DAY', time.strftime('%Y-%m-%d'))
EVAL_SIZE = int(os.environ.get('RESTORMER_NPU_V1_EVAL_SIZE', '256'))
CALIBRATION_COUNT = int(os.environ.get('RESTORMER_NPU_V1_CALIBRATION_COUNT', '32'))
VERIFY_ONNX = os.environ.get('RESTORMER_NPU_V1_VERIFY_ONNX', 'true').lower() in {'1', 'true', 'yes'}
CHECKPOINT_PATH = Path(os.environ['RESTORMER_NPU_V1_CHECKPOINT']).expanduser().resolve() if os.environ.get('RESTORMER_NPU_V1_CHECKPOINT') else None
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(255)

print('PACKAGE_ROOT', PACKAGE_ROOT)
print('PROJECT_ROOT', PROJECT_ROOT)
print('CONFIG_PATH', CONFIG_PATH)
print('DATA_ROOT', DATA_ROOT)
print('MODAL_DATA_VOLUME', MODAL_DATA_VOLUME)
print('MODAL_RUNS_VOLUME', MODAL_RUNS_VOLUME)
print('DEVICE', DEVICE)


In [ ]:
config = load_config(CONFIG_PATH)
profile_name, profile = resolve_profile(config, PROFILE)
data_summary = validate_data_layout(DATA_ROOT)
artifact_cfg = dict(config.get('artifacts', {}))
layout = build_run_layout(
    project_root=PROJECT_ROOT,
    run_name=RUN_NAME,
    started_day=STARTED_DAY,
    export_slug=str(artifact_cfg.get('export_slug', 'restormer_npu_v1')),
    onnx_name=str(artifact_cfg.get('onnx_name', 'restormer_npu_v1.onnx')),
    mxq_name=str(artifact_cfg.get('mxq_name', 'restormer_npu_v1.mxq')),
)
print(json.dumps({
    'profile_name': profile_name,
    'profile': profile,
    'train_pair_count': data_summary['train_pair_count'],
    'val_pair_count': data_summary['val_pair_count'],
    'run_root': str(layout.run_root),
    'export_root': str(layout.export_root),
}, indent=2))


## Baseline Export Audit

The default profile is `smoke`. Set `RESTORMER_NPU_V1_PROFILE=large` to export the full baseline profile.


In [ ]:
model = build_model(profile).to(DEVICE)
if CHECKPOINT_PATH is not None:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
    state_dict = checkpoint.get('model') or checkpoint.get('state_dict') or checkpoint
    model.load_state_dict(state_dict, strict=False)
contract = verify_model_contract(model, EVAL_SIZE, DEVICE)
onnx_summary = export_to_onnx(model, layout.onnx_path, DEVICE, EVAL_SIZE, VERIFY_ONNX)
audit = audit_onnx_graph(layout.onnx_path)
calibration = export_calibration_dataset(data_summary['train_pairs'], layout.calibration_dir, EVAL_SIZE, CALIBRATION_COUNT)
mxq_payload = run_mxq_handoff(PROJECT_ROOT, layout.onnx_path, layout.calibration_dir, layout.mxq_path, dry_run=True)

write_json(layout.operator_audit_path, audit)
write_json(layout.parity_path, onnx_summary['parity'])
write_json(layout.mxq_payload_path, mxq_payload)

display(Markdown(operator_risk_markdown(audit)))
print('Contract:')
print(json.dumps(contract, indent=2))
print('ONNX summary:')
print(json.dumps(onnx_summary, indent=2))
print('Calibration summary:')
print(json.dumps(calibration, indent=2))
print('MXQ handoff payload:')
print(json.dumps(mxq_payload, indent=2))
print('Artifacts:')
print(layout.onnx_path.resolve())
print(layout.operator_audit_path.resolve())
print(layout.parity_path.resolve())
print(layout.calibration_dir.resolve())
print(layout.mxq_payload_path.resolve())
print(layout.mxq_path.resolve())


## Appendix: Mitigations

This v1 notebook does not implement architecture sweeps. Future follow-up candidates to reduce operator risk include:

- replacing custom LayerNorm with InstanceNorm-style normalization
- replacing GELU with CELU-family activation
- removing the learnable temperature Softplus path
- replacing MDTA with a convolution-only mixing block

Those are design notes only in v1 and should be treated as separate code-change work.
